# ODS 层 — 原始数据入仓

将黄色 + 绿色出租车原始 Parquet 写入 ODS 层，保留原始字段，追加 `source_type` 和 `ingestion_ts` 元数据列，按 `source_type / pickup_year / pickup_month` 分区存储。

## 1. 初始化 SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ODS-Ingest")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE      = os.path.expanduser("~/tlc-spark-pipeline")
RAW_Y     = f"{BASE}/data/raw/yellow"
RAW_G     = f"{BASE}/data/raw/green"
ODS_Y     = f"{BASE}/data/warehouse/ods/yellow"
ODS_G     = f"{BASE}/data/warehouse/ods/green"

print(f"Spark {spark.version} ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/18 23:57:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 ready


## 2. 黄色出租车 → ODS

In [2]:
%%time
raw_yellow = spark.read.parquet(f"{RAW_Y}/*.parquet")

ods_yellow = (
    raw_yellow
    .withColumn("source_type",   F.lit("yellow"))
    .withColumn("ingestion_ts",  F.current_timestamp())
    # 统一时间字段名（为 DWD 统一做准备）
    .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    # 分区列
    .withColumn("pickup_year",  F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
)

print(f"Yellow raw rows: {raw_yellow.count():,}")
ods_yellow.printSchema()

Yellow raw rows: 9,554,778
root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- source_type: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = false)
 |-- pickup_year:

In [3]:
%%time
(
    ods_yellow
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(ODS_Y)
)
print(f"Yellow ODS written to: {ODS_Y}")

26/10/18 23:57:53 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Yellow ODS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ods/yellow
CPU times: user 13.8 ms, sys: 11.2 ms, total: 25 ms
Wall time: 20.7 s


## 3. 绿色出租车 → ODS

In [4]:
%%time
raw_green = spark.read.parquet(f"{RAW_G}/*.parquet")

ods_green = (
    raw_green
    .withColumn("source_type",   F.lit("green"))
    .withColumn("ingestion_ts",  F.current_timestamp())
    # 统一时间字段名
    .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("pickup_year",  F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
)

print(f"Green raw rows: {raw_green.count():,}")
ods_green.printSchema()

Green raw rows: 167,585
root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- source_type: string (nullable = false)
 |-- ingestion_ts: timestamp (n

In [5]:
%%time
(
    ods_green
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(ODS_G)
)
print(f"Green ODS written to: {ODS_G}")

Green ODS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ods/green
CPU times: user 4.84 ms, sys: 3.76 ms, total: 8.6 ms
Wall time: 1.26 s


## 4. 验证 ODS

In [6]:
for source, path in [("yellow", ODS_Y), ("green", ODS_G)]:
    df = spark.read.parquet(path)
    cnt = df.count()
    print(f"{source}: {cnt:,} rows")
    df.groupBy("pickup_year", "pickup_month").count().orderBy("pickup_year", "pickup_month").show()

spark.stop()
print("ODS ingest complete.")

yellow: 9,554,778 rows


+-----------+------------+-------+
|pickup_year|pickup_month|  count|
+-----------+------------+-------+
|       2002|          12|      4|
|       2008|          12|      1|
|       2009|           1|      4|
|       2023|          12|     10|
|       2024|           1|2964617|
|       2024|           2|3007533|
|       2024|           3|3582607|
|       2024|           4|      2|
+-----------+------------+-------+



green: 167,585 rows


+-----------+------------+-----+
|pickup_year|pickup_month|count|
+-----------+------------+-----+
|       2008|          12|    2|
|       2023|          12|    2|
|       2024|           1|56555|
|       2024|           2|53578|
|       2024|           3|57447|
|       2024|           4|    1|
+-----------+------------+-----+



ODS ingest complete.
